# Quantum AI — v16: manwithacat's 888-qubit DQI and chemistry refit, with our changes on top

This version collects, for each of the three tasks, the cheapest circuit we could verify with the unmodified
official verifiers at their full default shot counts, and adds our own improvements where we have them. Every file is
located by its SHA-256 and checked again after the copy; the verifiers then run inside this notebook.

| Task | Circuit | Qubits | Ratio (reference / cost) |
|---|---|---:|---:|
| invsqrt | yoyobar's 62-qubit circuit with one more Toffoli removed (ours, since v15) | 62 | 20.574 |
| dqi | manwithacat's 888-qubit splice with the chance-zero fix (2ed7ccf1) | 888 | 60.59 |
| dfthcsos | manwithacat's refit of heliosli's weights with our λ polish, per-slot loaders | 430 | 9.934 |

Geometric mean in our local run: **23.136**. The DQI ratio moves slightly with the verifier's seed (rare decoder
failures enter through the repetition factor R), so the leaderboard number can differ by a few tenths of a percent.


In [ ]:
import hashlib, json, math, os, pathlib, sys
os.environ.update(OMP_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1', MKL_NUM_THREADS='1')
import pandas as pd
from IPython.display import display

INPUT = pathlib.Path(os.environ.get('QCT_INPUT', '/kaggle/input'))
WORK = pathlib.Path(os.environ.get('QCT_WORK', '/kaggle/working'))
SUB = pathlib.Path(os.environ.get('QCT_SUB', '/tmp/v16_submission'))
SLUG = 'google-verifiable-quantum-circuit-compilation'
COMP = next(p for p in [INPUT / 'competitions' / SLUG, INPUT / SLUG] if (p / 'kickmix').exists())
DATA = next(INPUT.rglob('vqcc_v7.py')).parent
ROOTS = [p for p in INPUT.iterdir() if p.name not in (SLUG, 'competitions')]   # everything but the competition data
import lzma, shutil
XZ = pathlib.Path(os.environ.get('QCT_XZ', '/tmp/v16_xz'))
for i, p in enumerate(sorted(q for r in ROOTS for q in r.rglob('*.kmb.xz') if q.is_file())):   # xz-packed circuits: unpack once
    dst = XZ / str(i) / p.name[:-3]
    dst.parent.mkdir(parents=True, exist_ok=True)
    with lzma.open(p) as f, open(dst, 'wb') as g:
        shutil.copyfileobj(f, g, 1 << 22)
    print('unpacked', p.name, '->', dst, dst.stat().st_size // 2**20, 'MiB')
if XZ.exists():
    ROOTS.append(XZ)
WORK.mkdir(parents=True, exist_ok=True)
SUB.mkdir(parents=True, exist_ok=True)
WORKERS = int(os.environ.get('QCT_WORKERS', os.cpu_count() or 4))
REFERENCE = {'invsqrt': 2076774.233715282, 'dqi': 24439474177519.598, 'dfthcsos': 3355599588.3155365}
print('competition data:', COMP, '| v7 folder:', DATA, '| other inputs:', [p.name for p in ROOTS], '| workers:', WORKERS)


In [ ]:
sys.path.insert(0, str(DATA))
import vqcc_v7 as V
V.PINS = {   # output file -> SHA-256; the notebook pins every file itself
    'dfthcsos/angle_k_0.kmb': '9c31da0c1d047e7dcac638df807d3cef96eac6f736a8838a0ae6943e9c46c74f',
    'dfthcsos/angle_k_1.kmb': '39b8d10098be097f75148639ee302249bdabe112a9278a48565eadb0cb86810b',
    'dfthcsos/angle_k_10.kmb': 'e75f6b0685d754bf76f3f4d62589bf1d07b118faeeb0ab8745833733286db2fc',
    'dfthcsos/angle_k_11.kmb': '773e54f6f0d7c3f769db433f263d541cdcff1076fb75109740907e3249996141',
    'dfthcsos/angle_k_12.kmb': 'd86ca7ddd984059ded765b953cc936388517a7ca5789676ae90910a2be89769d',
    'dfthcsos/angle_k_13.kmb': '50d82cf65116f06c0fe469ebba312f72675b277263461ab88b4d15eca947c050',
    'dfthcsos/angle_k_14.kmb': 'ae2070998e31b7e05d01183914df20b6974c1760bddf90d1adac86ea75c7bf3f',
    'dfthcsos/angle_k_2.kmb': '2e8dfdeb69c496e9e510fe99722153663760980d45c35ad9b72e79ba799dde49',
    'dfthcsos/angle_k_3.kmb': '13817cf8d36ea112e62d7863edde2193aa5907dd0605612c4f0e1e54bacc173b',
    'dfthcsos/angle_k_4.kmb': '8dce8f4a380998d818bf8a08e4f3daa62aaeb31a25d084aa57549f931a937f3d',
    'dfthcsos/angle_k_5.kmb': '42cf7d76ef0f0b36640a9e65005214fdd4d9a9bb82b8cef6e658dbbdd8417099',
    'dfthcsos/angle_k_6.kmb': '7fd7bd333b1427ecf546b0c27717ed9428e255e5165c4db14e5f95f8af45c838',
    'dfthcsos/angle_k_7.kmb': '809a775b98836ee00e8783d3f616bb8517bb3a3621978bc704d20b543a8df4ec',
    'dfthcsos/angle_k_8.kmb': '42d0deb376981d0470bebdd370ca2adf79685b4bf3e6909be262e2beced030ec',
    'dfthcsos/angle_k_9.kmb': 'f1c49e7cf8e4f3a9adaffe167d2bbe21497a52eca2c7d09f765fc5a125e304c3',
    'dfthcsos/submission.npz': '47427ee032d3719931abeea0ac88f5a6656971eabd492b9d499cae1b42c6b34b',
    'dqi.kmb': '2ed7ccf189f13b71f6ff64ade4d6bbadd01bfe61e93c038a39158ccc83f29555',
    'invsqrt.kmb': '6dbb98b34c3b16f1c4e019350c10d42259a6faa97e464c9b7c9390f877f392f4',
}
ORIGIN = V.build(SUB, ROOTS)
for out, src in ORIGIN.items():
    print(f'{out:28s} <- {src}')
import subprocess, time
VER = COMP / 'verifiers'
ENV = dict(os.environ, PYTHONPATH=os.pathsep.join([str(COMP / 'kickmix'), str(VER)]))
COMMANDS = {   # the official verifiers, unmodified, at their default shot counts
    'invsqrt': [VER / 'invsqrt' / 'verify_submission_invsqrt.py', SUB / 'invsqrt.kmb', '--max-workers', str(WORKERS)],
    'dqi': [VER / 'dqi' / 'verify_submission_dqi.py', SUB / 'dqi.kmb'],
    'dfthcsos': [VER / 'dfthcsos' / 'verify.py', SUB / 'dfthcsos'],
}
RESULTS = {}
for task, cmd in COMMANDS.items():
    t0 = time.time()
    run = subprocess.run([sys.executable, '-u'] + [str(c) for c in cmd] + ['--salt', 'notebook-verification'],
                         capture_output=True, text=True, env=ENV)
    RESULTS[task] = json.loads(run.stdout[run.stdout.index('{'):]) if '{' in run.stdout else {'status': 'NO_OUTPUT', 'stderr': run.stderr[-2000:]}
    print(f"{task}: {RESULTS[task].get('status')} cost {RESULTS[task].get('score')} ({time.time() - t0:.0f} s)")
assert all(r.get('status') == 'PASS' for r in RESULTS.values()), {k: r.get('status') for k, r in RESULTS.items()}
(WORK / 'verification.json').write_text(json.dumps(RESULTS, indent=1, default=str))
import zipfile
with zipfile.ZipFile(WORK / 'submission.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(SUB.rglob('*')):
        if p.is_file():
            z.write(p, p.relative_to(SUB).as_posix())
print('submission.zip written,', (WORK / 'submission.zip').stat().st_size // 2**20, 'MiB')


## Results of this run

All values below come from the official verifiers in this run (salt `notebook-verification`).


In [ ]:
r = RESULTS
rows = [('Inverse square root', 'invsqrt'), ('DQI decoder', 'dqi'), ('FeMoCo lookups', 'dfthcsos')]
table = pd.DataFrame([{'Task': name, 'Qubits': r[k].get('num_qubits'), 'Toffoli (mean)': f"{r[k].get('toffoli_count', 0):,.0f}",
                       'Cost': f"{r[k]['score']:.5g}", 'Reference cost': f'{REFERENCE[k]:.5g}',
                       'Reference / cost': f"{REFERENCE[k] / r[k]['score']:.3f}"} for name, k in rows])
score = math.prod(REFERENCE[k] / r[k]['score'] for k in REFERENCE) ** (1 / 3)
display(table.style.hide(axis='index'))
print(f'Score (geometric mean of reference / cost): {score:.4f}')


## DQI: the 888-qubit decoder

The decoder is manwithacat's **2ed7ccf1** from [VQCC Quantum Voodoo](https://www.kaggle.com/code/manwithacat/vqcc-quantum-voodoo) (v4), used unchanged. It is a piece of *circuit surgery*, hengck23's idea: heliosli's pole release (each of nine one-bit pole flags is erased right after it is made and recomputed when needed) was translated wire by wire into SomeJoint's newer decoder 53d8b2, giving 889 qubits for +315 Toffolis; two zero tests that almost never fire were cut (hengck23's ideas); the recomputed pole, the 12-wire norm Sq(a) + a·b + x¹¹·Sq(b) and a small inverse were released inside the reverse pass to reach 888 qubits; and groups 4–7 of the extraction were rebuilt as homogeneous (N : D) chains that never invert, which removes the chance-zero inversions behind most failing shots and saves 1,460 Toffolis per shot.

This run: 0 value and 0 phase errors in 102,400 shots, 888 qubits, mean 429,377 Toffolis, ratio 60.593 (hengck23's 889-qubit iteration 014 in our v15: 59.842).

Our own DQI work this round was a search for a narrower circuit with measure-and-repay ("spooky") pebbling at the widest sites of hengck23's 889-qubit decoder, which shares this extraction design. It closed: the projective normalization of the extraction keeps two 24-bit words next to the 840 syndrome wires (48 private wires even with ghost moves), so 888 is the floor there, and a low-width division would cost more Toffolis than the qubit is worth. manwithacat reports the same wall for 2ed7ccf1 (888 → 887 is blocked by 24-bit multiplications that hold both inputs and the product).


## FeMoCo chemistry

The weights are manwithacat's constrained refit of heliosli's weights from [VQCC Quantum Voodoo](https://www.kaggle.com/code/manwithacat/vqcc-quantum-voodoo): λ_eff 48.3601 → 48.3007 with error₁, error₂ and the spectral gap held exactly on their limits (a projected L-BFGS extending anazemcev's constraint-manifold refit). Word table and loaders are unchanged: 15 ANGLE loaders assembled slot by slot by manwithacat (slots 2, 3 and 14 from Dmitrii Gluzdov, 0, 1, 4, 5 and 8 from heliosli, the rest from Atakan Aldemir's V3), on SomeJoint's team's (8, 143, 60) factorization.

**Our change: a λ polish with the word table and loaders fixed.** First a column gauge (Adam on a smoothed L1 objective; the physics is unchanged, Λ₂ 286.96696 → 286.95131), then the exact two-column Jacobi gauge (30 sweeps; λ_eff 48.30069 → 48.29927 with error₁, error₂ and the gap bit-for-bit unchanged), then a native inequality augmented-Lagrangian solve from our earlier work that spends the last ~0.0008 mHa of each error budget (λ_eff → 48.29564). Net −0.0105% chemistry cost, verified below.

Official verifier: λ_eff 48.29564, error₁ 19.99975 mHa, error₂ 19.99982 mHa, E_gap 3.9148990, T 16,266, Q 430, ratio 9.9337.

What we measured and closed on this shape (GPU runs on Kaggle): with unconstrained new 8-bit words, C = 56 ends at 20.44 mHa, C = 48 at 23.86 and C = 44 at 26.56, all over the 20 mHa gate; C = 60 passes at 19.48, but new words need new loaders, and the per-slot loaders exploit a Reed–Muller structure of these exact words (in each 16-address block the low bits of consecutive batches differ by an affine function of the four low address bits), which generic words lose.


## Inverse square root

The base is yoyobar's 62-qubit circuit ([notebook](https://www.kaggle.com/code/yoyobar/lb-22-99-invsqrt-101-62-qubits)): Akmal Xodarev (busyaprime)'s 63-qubit circuit, itself on mrmeows' hybrid line, with the encoded lookup, normalization and tail schedule reworked to free one qubit. Its 41 rotation angles and phase function are bit-identical to the 63-qubit circuit's on all 3,133,452 inputs.

**Our change (since v15): one history cache.** In the final shift of x back into place, the swap condition of `CSWAP q19 q6 q41` equals a fixed XOR of the swap condition of an earlier `CSWAP q16 q2 q23` and four live wires on every legal input, so the later CSWAP becomes two CNOTs from a cached wire that is then measured away with a Clifford-only repair. Mean 1,331.48 Toffolis, 62 qubits, cost 100,940.61 (yoyobar's 101,002.55), ratio 20.5742.

This round we tested two more ideas on this circuit and closed both: re-centring the lookup's constant term K0 per segment jointly with the 41 angles widens the phase margin only 1.33× (worth 0.15% at most), and merging the two phase passes needs 8 qubits that are not free (61 live qubits at the first phase pass, 35 of them the encoded lookup output). manwithacat independently searched all 1,501 Toffoli-class gates of this file for another affine identity and found none.


## Sources and credits

- **Portfolio:** manwithacat's [VQCC Quantum Voodoo](https://www.kaggle.com/code/manwithacat/vqcc-quantum-voodoo) (v4, 23.135 LB) supplies the DQI circuit and the chemistry weights; the rest continues our v15, whose base was alex chilton's [22.57 LB: Verified Three-Task Portfolio](https://www.kaggle.com/code/alexchilton/22-57-lb-verified-three-task-portfolio) (v59).
- **invsqrt:** yoyobar (62-qubit rework, made with GPT-6 Astra per its author), heliosli (the first 62-qubit circuit), Akmal Xodarev (busyaprime), mrmeows, and the lineage they credit (Phuc Thai / DePie, Dmitrii Gluzdov, marsyas). Ours: the one-Toffoli history cache.
- **dqi:** manwithacat (2ed7ccf1: the splice, the 888-qubit release and the chain failure fix, made with Claude per its author), SomeJoint / Agniv Banerjee's team (decoder 53d8b2 and the homogeneous-chain construction), heliosli (pole release, 4ee7a17b), hengck23 (circuit surgery and the flag-cut ideas; the 889-qubit projective decoder), Kirill Tymchishin, and the chain they credit: alex chilton, jday96314 (QuantumAI; no licence stated), mrmeows, Ahmed Berat Özer, Dmitrii Gluzdov, ViberTeam, Phuc Thai.
- **dfthcsos:** manwithacat (constrained λ refit, per-slot loader assembly), anazemcev (constraint-manifold refit), heliosli (weights, loaders), Dmitrii Gluzdov and Atakan Aldemir (loaders, word table), SomeJoint's team (R8/B143/C60 factorization), Sargis Khachatryan, sigmaborov, baidalinadilzhan, Ahmed Berat Özer, Michael Rustad, mrmeows, lightningv08, marculera, kozykappa. Ours: the λ polish described above.

Organizers: task definitions, verifiers and kickmix
([competition](https://www.kaggle.com/competitions/google-verifiable-quantum-circuit-compilation)).
